<a href="https://colab.research.google.com/github/BlancaHoma14/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion20/sesion20_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 20 · Hito 2: Analizador Semántico Completo y Scopes

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Puede el compilador responder de una sola pasada quién es cada nombre en un programa con bloques y funciones?
>
> **Objetivo de la sesión:** Integrar la tabla de símbolos con el árbol de ámbitos para resolver nombres con shadowing y contar los símbolos visibles desde un punto del programa.

**Entregable:** tus funciones profundidad_de() y contar_visibles() en artesano_sesion20_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion20.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes una cadena de ámbitos con sus símbolos; profundidad_de() dice a cuántos niveles está el nombre (0 es el ámbito actual) y contar_visibles() cuenta los nombres distintos que se ven desde el bloque actual. El harness arma cadenas de tres niveles con nombres repetidos.

Al final de la sesión tendrás: **Un resolvedor de nombres con profundidad y un contador de símbolos visibles que respeta shadowing, con el harness del Hito 2 en verde.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 20](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion20.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion20.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion20.h"

int profundidad_de(const Ambito *actual, const char *nombre);
int contar_visibles(const Ambito *actual);
```

- profundidad_de() devuelve -1 si el nombre no existe y -2 si algún argumento es NULL.
- contar_visibles() cuenta cada nombre una sola vez, aunque esté en varios ámbitos.
- El ámbito más interno siempre gana: el shadowing no se promedia ni se suma.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion20_c.c` junto con `artesano_sesion20_c.c`
> (o `ia_sesion20_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `nombre en el ambito actual` | 0 |
| 2 | `nombre dos niveles arriba` | 2 |
| 3 | `shadowing gana el interno` | 0 |
| 4 | `nombre no visible` | -1 |
| 5 | `visibles sin repetir` | 3 |
| 6 | `ambito NULL` | -2 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 20 - Resolucion de nombres y visibilidad ===
Caso                               esperado:              obtenido:              resultado
[ 1] nombre en el ambito actual         esperado: 0            obtenido: 0            OK
[ 2] nombre dos niveles arriba          esperado: 2            obtenido: 2            OK
[ 3] shadowing gana el interno          esperado: 0            obtenido: 0            OK
[ 4] nombre no visible                  esperado: -1           obtenido: -1           OK
[ 5] visibles sin repetir               esperado: 3            obtenido: 3            OK
[ 6] ambito NULL                        esperado: -2           obtenido: -2           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion20.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] profundidad_de() recorre la cadena completa y devuelve el nivel del ámbito más cercano que declara el nombre.
- [ ] contar_visibles() elimina duplicados incluso cuando el mismo nombre aparece en dos niveles.
- [ ] Los argumentos NULL se reportan con -2 sin desreferenciar memoria inválida.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion20_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion20.h
/* api_sesion20.h - contrato de la Sesion 20
 * Resolución de nombres y visibilidad
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION20_H
#define API_SESION20_H

#define AMBITO_MAX 8
typedef struct Ambito {
    const char *nombre;
    const char *simbolos[AMBITO_MAX];
    int total;
    struct Ambito *padre;
} Ambito;

int profundidad_de(const Ambito *actual, const char *nombre);
int contar_visibles(const Ambito *actual);

#endif /* API_SESION20_H */


Writing api_sesion20.h


In [3]:
%%writefile artesano_sesion20_c.c

/* artesano_sesion20_c.c - Fase 1: El Artesano */

#include "api_sesion20.h"
#include <string.h>

/* Busca un nombre en un ambito */
static int existe_en_ambito(const Ambito *a,
                            const char *nombre) {
    if (a == NULL || nombre == NULL) {
        return 0;
    }

    int limite = a->total;

    if (limite < 0) limite = 0;
    if (limite > AMBITO_MAX) limite = AMBITO_MAX;

    for (int i = 0; i < limite; i++) {
        if (a->simbolos[i] != NULL &&
            strcmp(a->simbolos[i], nombre) == 0) {
            return 1;
        }
    }

    return 0;
}

/* Devuelve el nivel donde se encuentra el nombre */
int profundidad_de(const Ambito *actual,
                   const char *nombre) {

    if (actual == NULL || nombre == NULL) {
        return -2;
    }

    int profundidad = 0;

    for (const Ambito *p = actual;
         p != NULL;
         p = p->padre) {

        if (existe_en_ambito(p, nombre)) {
            return profundidad;
        }

        profundidad++;
    }

    return -1;
}

/* Cuenta los nombres visibles sin repetir */
int contar_visibles(const Ambito *actual) {

    if (actual == NULL) {
        return -2;
    }

    int contador = 0;

    for (const Ambito *p = actual;
         p != NULL;
         p = p->padre) {

        int limite = p->total;

        if (limite < 0) limite = 0;
        if (limite > AMBITO_MAX) limite = AMBITO_MAX;

        for (int i = 0; i < limite; i++) {

            const char *nombre = p->simbolos[i];

            if (nombre == NULL) {
                continue;
            }

            int repetido = 0;

            /* Revisar los ambitos mas internos */
            for (const Ambito *q = actual;
                 q != p;
                 q = q->padre) {

                if (existe_en_ambito(q, nombre)) {
                    repetido = 1;
                    break;
                }
            }

            /* Revisar duplicados anteriores del mismo ambito */
            for (int j = 0; j < i && !repetido; j++) {
                if (p->simbolos[j] != NULL &&
                    strcmp(p->simbolos[j], nombre) == 0) {
                    repetido = 1;
                }
            }

            if (!repetido) {
                contador++;
            }
        }
    }

    return contador;
}


Writing artesano_sesion20_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion20_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


**Contrato C-R-E-O**

**Contexto (C):** Estoy trabajando en Lenguajes y Autómatas II, Sesión 20, utilizando lenguaje C11 y el compilador GCC. Tengo una estructura Ambito que contiene un nombre, un arreglo de símbolos con capacidad máxima AMBITO_MAX, un contador de símbolos y un puntero al ámbito padre.

**Rol (R):** Actúa como un ingeniero de software especializado en compiladores, análisis semántico, tablas de símbolos y administración de ámbitos.

**Especificación (E):** Implementa las funciones profundidad_de() y contar_visibles() respetando exactamente las firmas establecidas en api_sesion20.h.

profundidad_de() debe buscar un nombre desde el ámbito actual hacia sus padres.

Debe devolver 0 si el nombre está en el ámbito actual.

Debe devolver el nivel correspondiente si está en un ámbito superior.

Debe devolver -1 si el nombre no existe.

Debe devolver -2 cuando alguno de sus argumentos sea NULL.

contar_visibles() debe contar cada nombre una sola vez.

Debe respetar el shadowing, dando prioridad al ámbito más interno.

Debe evitar accesos fuera de los límites del arreglo simbolos.

Debe controlar nombres NULL y evitar desreferenciar punteros inválidos.

No debe modificar las estructuras originales ni utilizar main().

Debe compilar con -std=c11 -Wall -Wextra -Werror.

**Output (O):** Entrega el código completo y modular en C, respetando las firmas originales, con comentarios explicativos y un análisis de complejidad temporal y espacial. No modifiques api_sesion20.h ni el contrato del harness.

In [4]:
%%writefile ia_sesion20_c.c

/* ia_sesion20_c.c - Fase 2: El Cientifico
 * Resolucion de nombres y visibilidad.
 */

#include "api_sesion20.h"
#include <string.h>

/* Comprueba si un nombre existe en un ambito */
static int buscar_simbolo(const Ambito *ambito,
                          const char *nombre) {

    if (ambito == NULL || nombre == NULL) {
        return 0;
    }

    int total = ambito->total;

    if (total < 0) {
        total = 0;
    }

    if (total > AMBITO_MAX) {
        total = AMBITO_MAX;
    }

    for (int i = 0; i < total; i++) {

        const char *simbolo = ambito->simbolos[i];

        if (simbolo != NULL &&
            strcmp(simbolo, nombre) == 0) {
            return 1;
        }
    }

    return 0;
}

/* Busca el nombre y devuelve su profundidad */
int profundidad_de(const Ambito *actual,
                   const char *nombre) {

    if (actual == NULL || nombre == NULL) {
        return -2;
    }

    int nivel = 0;

    const Ambito *ambito = actual;

    while (ambito != NULL) {

        if (buscar_simbolo(ambito, nombre)) {
            return nivel;
        }

        ambito = ambito->padre;
        nivel++;
    }

    return -1;
}

/* Cuenta los nombres diferentes que son visibles */
int contar_visibles(const Ambito *actual) {

    if (actual == NULL) {
        return -2;
    }

    int total_visibles = 0;

    const Ambito *ambito = actual;

    while (ambito != NULL) {

        int total = ambito->total;

        if (total < 0) {
            total = 0;
        }

        if (total > AMBITO_MAX) {
            total = AMBITO_MAX;
        }

        for (int i = 0; i < total; i++) {

            const char *nombre = ambito->simbolos[i];

            if (nombre == NULL) {
                continue;
            }

            int oculto = 0;

            /* Comprobar si un ambito mas interno
               ya declara el mismo nombre */
            const Ambito *interno = actual;

            while (interno != ambito) {

                if (buscar_simbolo(interno, nombre)) {
                    oculto = 1;
                    break;
                }

                interno = interno->padre;
            }

            /* Comprobar si ya se conto en este ambito */
            for (int j = 0; j < i && !oculto; j++) {

                const char *anterior = ambito->simbolos[j];

                if (anterior != NULL &&
                    strcmp(anterior, nombre) == 0) {
                    oculto = 1;
                }
            }

            if (!oculto) {
                total_visibles++;
            }
        }

        ambito = ambito->padre;
    }

    return total_visibles;
}


Writing ia_sesion20_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion20_c.c
/* harness_sesion20_c.c - auditoria automatica de la Sesion 20
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion20_c.c artesano_sesion20_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion20.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 20 - Resolucion de nombres y visibilidad ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; g.simbolos[g.total++] = "x"; Ambito l; memset(&l, 0, sizeof l); l.nombre = "bloque"; l.padre = &g; l.simbolos[l.total++] = "y"; int r = profundidad_de(&l, "y"); verificar_int("nombre en el ambito actual", (long)(0), (long)(r)); }
    { Ambito raiz; memset(&raiz, 0, sizeof raiz); raiz.nombre = "raiz"; raiz.simbolos[raiz.total++] = "x"; Ambito medio; memset(&medio, 0, sizeof medio); medio.nombre = "medio"; medio.padre = &raiz; Ambito hoja; memset(&hoja, 0, sizeof hoja); hoja.nombre = "hoja"; hoja.padre = &medio; int r = profundidad_de(&hoja, "x"); verificar_int("nombre dos niveles arriba", (long)(2), (long)(r)); }
    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; g.simbolos[g.total++] = "x"; Ambito l; memset(&l, 0, sizeof l); l.nombre = "bloque"; l.padre = &g; l.simbolos[l.total++] = "x"; int r = profundidad_de(&l, "x"); verificar_int("shadowing gana el interno", (long)(0), (long)(r)); }
    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; g.simbolos[g.total++] = "x"; Ambito l; memset(&l, 0, sizeof l); l.nombre = "bloque"; l.padre = &g; int r = profundidad_de(&l, "z"); verificar_int("nombre no visible", (long)(-1), (long)(r)); }
    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; g.simbolos[g.total++] = "x"; g.simbolos[g.total++] = "y"; Ambito l; memset(&l, 0, sizeof l); l.nombre = "bloque"; l.padre = &g; l.simbolos[l.total++] = "y"; l.simbolos[l.total++] = "z"; int r = contar_visibles(&l); verificar_int("visibles sin repetir", (long)(3), (long)(r)); }
    { int r = profundidad_de(NULL, "x"); verificar_int("ambito NULL", (long)(-2), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion20.md" "\n");
    return 0;
}


Writing harness_sesion20_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion20_c.c artesano_sesion20_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 20 - Resolucion de nombres y visibilidad ===
Caso                               esperado:              obtenido:              resultado
[ 1] nombre en el ambito actual         esperado: 0            obtenido: 0            OK
[ 2] nombre dos niveles arriba          esperado: 2            obtenido: 2            OK
[ 3] shadowing gana el interno          esperado: 0            obtenido: 0            OK
[ 4] nombre no visible                  esperado: -1           obtenido: -1           OK
[ 5] visibles sin repetir               esperado: 3            obtenido: 3            OK
[ 6] ambito NULL                        esperado: -2           obtenido: -2           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion20.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion20_c.c ia_sesion20_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 20 - Resolucion de nombres y visibilidad ===
Caso                               esperado:              obtenido:              resultado
[ 1] nombre en el ambito actual         esperado: 0            obtenido: 0            OK
[ 2] nombre dos niveles arriba          esperado: 2            obtenido: 2            OK
[ 3] shadowing gana el interno          esperado: 0            obtenido: 0            OK
[ 4] nombre no visible                  esperado: -1           obtenido: -1           OK
[ 5] visibles sin repetir               esperado: 3            obtenido: 3            OK
[ 6] ambito NULL                        esperado: -2           obtenido: -2           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion20.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion20.md` a `auditorias/auditoria_sesion20.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
